### 미니프로젝트1
- 공공데이터 기반 지역별 통계 분석

#### 가설
등록인구 대비 화장실 밀도가 유독 낮은 지역은, 실제로는 오피스/상업지구·관광지일 가능성이 높을 것이다.

- 이 같은 차이는 지역 특성에 따른 것으로 분석된다. 인구가 많은 도시는 상업시설이나 민간 건물 화장실 이용 비중이 높았다.
[출처] 경기신문 (https://www.kgnews.co.kr/news/article.html?no=905844)

- 하위 가설: 
1. 고령 인구가 많을수록 장애인 대변기수의 비율이 높을 것이다./ 영유아인구가 많을수록 유아 대변기수의 비율이 높을 것이다. 
2. 인구가 적을수록 인구대비 여자화장실(or 대변기수)가 적을 것이다. 
3. 영유아 인구가 많을수록 기저귀 교환대가 많을 것이다.

In [ ]:
#상대 경로 - 파일 열고 구조 확인 후 변수에 넣기
import pandas as pd
import re
import sqlite3

In [ ]:
# 파일읽기
toilet = pd.read_csv(r'..\data\raw\공중화장실정보.csv', encoding= 'cp949')
pop = pd.read_csv(r'..\data\raw\주민등록인구수_20260630.csv', encoding= 'cp949')
print("화장실 데이터:", toilet.shape)
print("인구 데이터:", pop.shape)

화장실 데이터: (53552, 34)
인구 데이터: (3618, 230)


In [ ]:
#시군구 
def parse_sigungu(addr):
    m = re.match(r'^(\S+[도특별시광역시자치시])\s+(\S+[시군구])(\s+\S+[구])?', str(addr))
    if m:
        return m.group(1), (m.group(2) + (m.group(3) or '')).strip()
    return None, None

In [ ]:
# 원본 유지하면서 파생 컬럼 추가
toilet['total_seats'] = (
    toilet['남성용-대변기수'] + toilet['남성용-소변기수'] + 
    toilet['남성용-장애인용대변기수'] + toilet['남성용-장애인용소변기수'] +
    toilet['남성용-어린이용대변기수'] + toilet['남성용-어린이용소변기수'] +
    toilet['여성용-대변기수'] + 
    toilet['여성용-장애인용대변기수'] + toilet['여성용-어린이용대변기수']
)

#하위 가설을 위해 장애인, 유아 대변기수 분리
toilet['disabled_seats'] = (
    toilet['남성용-장애인용대변기수'] + toilet['남성용-장애인용소변기수'] +
    toilet['여성용-장애인용대변기수']
)

toilet['child_seats'] = (
    toilet['남성용-어린이용대변기수'] + toilet['남성용-어린이용소변기수'] +
    toilet['여성용-어린이용대변기수']
)

# 하위 가설을 위해 기저귀 교환대 유무 분리
# 2. Y/N → Boolean
toilet['has_diaper_table'] = toilet['기저귀교환대유무'].map({'Y': True, 'N': False})

# 3. 지역별로 개수 + 비율 같이 보기
result = toilet.groupby('sigungu')['has_diaper_table'].agg(
    total_toilets='count',       # 해당 지역 화장실 총 개수
    diaper_count='sum',          # 기저귀교환대 있는 화장실 개수
    diaper_rate='mean'           # 설치율 (0~1 비율)
).reset_index()

result['diaper_rate_pct'] = (result['diaper_rate'] * 100).round(1)

print(result.sort_values('diaper_rate_pct', ascending=False).head(10))

      sigungu  total_toilets  diaper_count  diaper_rate  diaper_rate_pct
269   춘천시 강촌구              1             1     1.000000            100.0
242    중구 용산구              1             1     1.000000            100.0
228  전주시  완산구              2             2     1.000000            100.0
49        권선구              1             1     1.000000            100.0
226       전주시              4             3     0.750000             75.0
147       수원시              3             2     0.666667             66.7
144    수성구 달구             30            18     0.600000             60.0
42     구리시 동구              7             4     0.571429             57.1
200   용인시 기흥구            161            86     0.534161             53.4
175       연수구            184            96     0.521739             52.2


In [ ]:
# 5. 파생 컬럼 (변기 수 합계)
toilet['총변기수'] = toilet[['남성용대변기수','남성용소변기수','여성용대변기수']].sum(axis=1)

0.983567373767553


In [ ]:
# 6. 결측치 제거
toilet = toilet.dropna(subset=['시군구명'])

In [ ]:
conn = sqlite3.connect("mini_project.db")
toilet.to_sql("toilet", conn, if_exists="replace", index=False)
pop.to_sql("population", conn, if_exists="replace", index=False)

In [ ]:
set(toilet['sigungu']) - set(pop['sigungu'])  # 화장실엔 있는데 인구엔 없는 지역명
set(pop['sigungu']) - set(toilet['sigungu'])  # 반대

In [ ]:
query = """
SELECT t.시군구명, COUNT(*) AS 화장실개수, SUM(t.총변기수) AS 변기총수, p.총인구수
FROM toilet t
JOIN population p ON t.시군구명 = p.시군구명
GROUP BY t.시군구명
"""
result = pd.read_sql(query, conn)

In [ ]:
import pandas as pd
import re

# 1. 파일 읽기 (인코딩: CP949)
toilet = pd.read_csv('../data/raw/공중화장실정보.csv', encoding='cp949')
pop = pd.read_csv('../data/raw/주민등록인구수_20260630.csv', encoding='cp949')

print("화장실 원본:", toilet.shape)
print("인구 원본:", pop.shape)


# ===================================================
# 2. 화장실 데이터 전처리
# ===================================================

# 2-1. 시군구 파싱 (도로명주소 기준)
def parse_sigungu(addr):
    if pd.isna(addr):
        return None
    m = re.match(r'^\S+?[도특별시광역시자치시]\s+(\S+?[시군구](\s+\S+?구)?)', addr)
    return m.group(1).strip() if m else None

toilet['sigungu'] = toilet['소재지도로명주소'].apply(parse_sigungu)

# 2-2. 변기수 파생 컬럼 (성인/장애인/어린이 구분해서 저장)
toilet['male_seats'] = (
    toilet['남성용-대변기수'] + toilet['남성용-소변기수']
)
toilet['female_seats'] = toilet['여성용-대변기수']

toilet['disabled_seats'] = (
    toilet['남성용-장애인용대변기수'] + toilet['남성용-장애인용소변기수'] +
    toilet['여성용-장애인용대변기수']
)
toilet['child_seats'] = (
    toilet['남성용-어린이용대변기수'] + toilet['남성용-어린이용소변기수'] +
    toilet['여성용-어린이용대변기수']
)
toilet['total_seats'] = (
    toilet['male_seats'] + toilet['female_seats'] +
    toilet['disabled_seats'] + toilet['child_seats']
)

# 2-3. 기저귀교환대 Y/N → 0/1
toilet['has_diaper_table'] = (toilet['기저귀교환대유무'] == 'Y').astype(int)

# 2-4. 필요한 컬럼만 최종 선택
toilet_clean = toilet[[
    '관리번호', 'sigungu',
    'male_seats', 'female_seats', 'disabled_seats', 'child_seats', 'total_seats',
    'has_diaper_table'
]].rename(columns={'관리번호': 'toilet_id'})

# 2-5. 결측/중복 체크
print("시군구 파싱 실패:", toilet_clean['sigungu'].isna().sum())
print("toilet_id 중복:", toilet_clean['toilet_id'].duplicated().sum())

toilet_clean = toilet_clean.dropna(subset=['sigungu'])


# ===================================================
# 3. 인구 데이터 전처리 (읍면동 → 시군구 단위로 집계)
# ===================================================

# 3-1. 나이대별 컬럼 분리 (고령: 65세 이상, 유아: 0~6세)
male_cols = [c for c in pop.columns if c.endswith('세남자') or c == '100세이상 남자']
female_cols = [c for c in pop.columns if c.endswith('세여자') or c == '100세이상 여자']

elderly_male_cols = [c for c in male_cols if any(str(a) in c for a in range(65, 111))]
elderly_female_cols = [c for c in female_cols if any(str(a) in c for a in range(65, 111))]
child_male_cols = [f'{a}세남자' for a in range(0, 7)]
child_female_cols = [f'{a}세여자' for a in range(0, 7)]

pop['elderly_pop'] = pop[elderly_male_cols].sum(axis=1) + pop[elderly_female_cols].sum(axis=1)
pop['child_pop'] = pop[child_male_cols].sum(axis=1) + pop[child_female_cols].sum(axis=1)

# 3-2. 읍면동 단위 → 시군구 단위로 집계 (groupby sum)
population_clean = pop.groupby(['시도명', '시군구명']).agg(
    male_pop=('남자', 'sum'),
    female_pop=('여자', 'sum'),
    total_pop=('계', 'sum'),
    elderly_pop=('elderly_pop', 'sum'),
    child_pop=('child_pop', 'sum')
).reset_index().rename(columns={'시도명': 'sido', '시군구명': 'sigungu'})

print("인구 시군구 집계 후:", population_clean.shape)


# ===================================================
# 4. 매칭 검증 (JOIN 전에 반드시 확인)
# ===================================================

toilet_sigungu = set(toilet_clean['sigungu'])
pop_sigungu = set(population_clean['sigungu'])

print("매칭률:", toilet_clean['sigungu'].isin(pop_sigungu).mean())
print("화장실에만 있는 지역명:", list(toilet_sigungu - pop_sigungu)[:15])